# ML Income Classification Example

As mentioned in the README.md, we'll be following the ML Experimentation Life Cycle laid out in __Design an ML System (from Scratch)__

![ML Exp Life Cycle](./images/ML_experiment_life_cycle.png)

Fortunately, since this is a practice example, the first step, "Problem Formulation" has already been done for us. Normally, this would be the result of looking at a business problem and deciding that ML provides a good solution, but in this case, we know that we want to take the provided training data (in `data/adult.names`) which includes a variety of values and find a model to predict whether the person whose features are represented by the values in a particular row of data has an annual income of \\$50k or less or not (that is, their annual income is above \\$50k).

So, let's move on to Exploratory Data Analysis (EDA). This actually requires code…

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


Once we load the data into a dataframe, we can get an idea of what it looks like

In [ ]:
# Load the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
column_names = [
    "age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
    "hours_per_week", "native_country", "income"
]

df = pd.read_csv(url, names=column_names, na_values=" ?", skipinitialspace=True)

# Preview the data
print("Initial data shape:", df.shape) # a tuple (rows, columns)
print(df.head()) # first 5 rows of the data

and get an idea of how the overall data is structured

In [ ]:
print(df.describe()) # summary statistics for numeric columns
print(df.info()) # data types and completeness counts 

Some quick visualizations

In [ ]:
# Visualizations
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='income')
plt.title('Income Distribution')
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(x='income', y='age', data=df)
plt.title('Age Distribution by Income Group')
plt.show()

In [ ]:
df['fnlwgt'].plot.hist()

In [ ]:
df['education_num'].plot.hist()

In [ ]:
df['capital_gain'].plot.hist()

In [ ]:
df['capital_loss'].plot.hist()

In [ ]:
df['hours_per_week'].plot.hist()

## Feature selection
Let's separate the features (variables we'll base the prediction on) from the target (a variable that we're trying to predict). In addition, we'll break the features and targets into a sets we can train the model on and sets we can use to test the model once the training is done.

In [ ]:
# Feature selection
X = df.drop("income", axis=1)
y = df["income"].apply(lambda x: 1 if x == ">50K" else 0)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



## Data clean up
There are multiple non-numeric, categorical columns. ML algorithms can only work with numbers so we can convert these to numeric values. We'll also want to standardize the numeric values since ML models can be screwed up by data that's on very different scales

In [ ]:
# Identify categorical and numeric columns
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

In [ ]:
# Create a pipeline to apply the preprocessors that encode the non-numeric data
# Preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

In [ ]:
# Then use that with two different classifiers so we can generate different 
# models for comparison

# Define models
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000)
}

results = {}

## Build ("fit") the models and make predictions
We'll apply the pipelines we defined above to create our models on the preprocessed data

In [ ]:
# Train and evaluate each model
for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    results[name] = {
        "model": model,
        "pipeline": pipeline,
        "y_pred": y_pred,
        "y_proba": y_proba,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba)
    }

    print(f"{name} Classifier:")
    print(classification_report(y_test, y_pred))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
    plt.title(f"{name} Confusion Matrix")
    plt.show()


## Evaluate and compare models

In [ ]:
# === Plot ROC curves ===
plt.figure(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["y_proba"])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {res['roc_auc']:.2f})")

plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

# === Side-by-side bar plot of metrics ===
metrics_df = pd.DataFrame({
    model: {
        "Accuracy": res["accuracy"],
        "Precision": res["precision"],
        "Recall": res["recall"],
        "F1 Score": res["f1"],
        "ROC AUC": res["roc_auc"]
    }
    for model, res in results.items()
}).T

metrics_df.plot(kind="bar", figsize=(10, 6))
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0.0, 1.1)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.grid(axis="y")
plt.tight_layout()
plt.show()